# The Latency Budget

Simulate one phone turn and move the knobs. Predict p50 and p95 before each run.

In [ ]:
import sys
sys.path.insert(0, '../code')
from dataclasses import replace
import numpy as np
import main

for name, cfg in main.optimization_ladder():
    s = main.summarize(main.simulate_turns(cfg, n=10_000)['total'])
    print(f"{name:<30} p50 {s['p50']:5.0f} ms  p95 {s['p95']:5.0f} ms  under 1 s {s['under_1s']:.0%}")

## Your own pipeline

Replace the medians and 95th percentiles with numbers from your providers. Which single change moves p95 the most?

In [ ]:
mine = replace(main.PipelineConfig(),
               end_of_turn=main.Stage('end of turn', 450, 900),
               llm_ttft=main.Stage('llm ttft', 380, 900),
               provider_rtt_ms=40.0,
               preemptive=True)
result = main.simulate_turns(mine, n=10_000)
print({k: round(float(np.median(result[k]))) for k in main.STAGE_ORDER})
print(main.summarize(result['total']))

## Chunking the first sentence

Rewrite the reply so that it starts with a short acknowledgement. How much earlier does the first audio arrive?

In [ ]:
long_first = 'Đơn hàng số bốn bảy hai một của anh đã được xác nhận và sẽ được giao vào sáng thứ năm tuần này. Anh cần gì thêm không?'
short_first = 'Dạ, em kiểm tra rồi ạ. Đơn bốn bảy hai một sẽ giao sáng thứ năm. Anh cần gì thêm không?'
for reply in (long_first, short_first):
    print(main.split_for_tts(reply)[0], '->', round(main.first_audio_ms(reply, True, True)), 'ms')